## DSAN 6000 Homework 3A: Allocating Tasks to Parallel Workers with `joblib`

## Overview

You made it to the first DSAN 6000 homework introducing a new coding concept! The goal of this part is for you to gain hands-on experience using **`joblib`** to quickly parallelize an **embarrassingly-parallel** task.

In this case, the problem is one that relates back to Week 1 and Week 2 content on the **OnLine Transaction Processing (OLTP)** mode of data collection and processing: the setup is that you run a website where users from across the globe can pay for products, therefore necessitating **currency conversion** from the currency in which the user paid to the currency that your business uses. In this case, your company operates in the [Eurozone](https://en.wikipedia.org/wiki/Eurozone), so your task will be to use **parallel processing** to see how/whether you can "push" your OLTP system to process a large volume of transactions in a short amount of time.

Thus, this basic task was chosen specifically so that you can take your intuitions about how long it might take for a **serial** algorithm to carry out these conversions, and compare with how quickly you'll be able to complete it using `joblib` to **distribute** the subtasks to different **workers**, who can process the currencies and amounts in parallel.

## Part 1: Loading and Preparing Data

### Part 1.1: Use `boto3` to Download the `.parquet` File From S3

Since you already figured out how to use `boto3` to connect to and download files from an S3 bucket in HW2, here we have provided code for you importing `boto3` and setting up the `s3` client object. Your task is to use this client to **download** the file at the following URI:

```
s3://dsan6000-data/transactions_10m.parquet
```

To your EC2 instance, saving it to have the same filename in a `data` subfolder (within the `dsan6000-hw03-parallel-processing` folder). We have already included a `.gitignore` file in the template repo, to ensure that this `.parquet` file doesn't get pushed to GitHub, since it is larger than the allowed invidual file size on GitHub!

In [1]:
#| label: q1.1-init
import boto3
s3 = boto3.client('s3')

In [2]:
#| label: q1.1-response
# Your code here: Download transactions_10m.parquet to the data subfolder
s3.download_file('dsan6000-data', 'transactions_10m.parquet', 'data/transactions_10m.parquet')

### Part 1.2: Loading the Data Into Pandas

In [3]:
#| label: q1.2-init
import pandas as pd

In [4]:
#| label: q1.2-response
oltp_df = pd.read_parquet("data/transactions_10m.parquet")
oltp_df

,timestamp,customer_id,product_id,amount
0,2026-08-19 23:02:45.546837,10658,78,71.01 PLN
1,2026-08-19 23:02:52.713547,59832,69,86.03 THB
2,2026-08-19 23:02:52.925565,83350,81,27.47 BGN
3,2026-08-19 23:02:53.636365,33223,18,69.47 JPY
4,2026-08-19 23:02:59.583317,31854,37,26.29 KRW
...,...,...,...,...
9999995,2026-09-18 23:06:08.793219,5274,28,58.27 AUD
9999996,2026-09-18 23:06:19.618683,12165,57,34.30 KRW
9999997,2026-09-18 23:06:22.577753,79851,85,33.57 HUF
9999998,2026-09-18 23:06:23.787475,38173,69,86.17 JPY


## Part 2: Converting Currencies in Serial

In [10]:
import time
disp_time = lambda start, end: print('{:.4f} s'.format(end - start))

In [ ]:
from currency_converter import CurrencyConverter
converter = CurrencyConverter()

In [40]:
def convert_currency(amount):
  currency_elts = amount.split(" ")
  num_val = float(currency_elts[0])
  ccode = currency_elts[1]
  return converter.convert(num_val, currency=ccode, new_currency='EUR')

In [41]:
convert_currency('93.60 USD')

81.03194528612241

In [ ]:
amount_vals = oltp_df['amount'].to_list()

: 

In [ ]:
serial_start = time.time()
amounts_converted = [convert_currency(a) for a in amount_vals]
serial_end = time.time()
disp_time(serial_start, serial_end)

In [ ]:
len(amounts_converted)

10000000

## Part 3: Converting Currencies in Parallel

In [5]:
import joblib

In [6]:
joblib.cpu_count()

2

In [7]:
import numpy as np

In [11]:
class MyConverter:
  def __init__(self):
    self.conversion_rates = {
      'USD': 0.865726, 'JPY': 0.005602, 'CZK': 0.041162, 'DKK': 0.133774,
      'GBP': 1.168252, 'HUF': 0.002737, 'PLN': 0.230319, 'RON': 0.190230,
      'SEK': 0.088645, 'CHF': 1.060333, 'ISK': 0.007153, 'NOK': 0.092876,
      'TRY': 0.017805, 'AUD': 0.617208, 'BRL': 0.167887, 'CAD': 0.623403,
      'CNY': 0.129051, 'HKD': 0.110376, 'IDR': 0.000049, 'INR': 0.009060,
      'KRW': 0.000643, 'MXN': 0.050710, 'MYR': 0.212395, 'NZD': 0.499700,
      'PHP': 0.013771, 'SGD': 0.681385, 'THB': 0.026037, 'ZAR': 0.053278,
    }

  def convert(self, amount: str):
    currency_elts = amount.split(" ")
    num_val = float(currency_elts[0])
    ccode = currency_elts[1]
    return self.conversion_rates[ccode] * num_val


In [13]:
amount_vals = oltp_df['amount'].to_list()

In [14]:
parallel_runner = joblib.Parallel(n_jobs=-1, batch_size=10000)
par_start = time.time()
amounts_cleaned_parallel = parallel_runner(
  joblib.delayed(MyConverter().convert)(a) for a in amount_vals
)
par_end = time.time()
disp_time(par_start, par_end)

KeyError: 'BGN'